# TE-PAI vs. Trotter — snapshot estimation with error bars

We evolve a **7-qubit Heisenberg spin chain** from a Néel state and estimate
$\langle Z_0(t)\rangle$:

1. **Exact** — statevector expectation of the first-order Trotter circuit (reference line).
2. **Trotter** — the (deep) Trotter circuit measured with `Ns` shots.
3. **TE-PAI** — `M` shallow random circuits, signed-weighted; evaluated **in parallel across all CPU cores** via `TEPAI.estimate`.

Both are unbiased for the Trotter-evolved value, so they track the exact curve within error bars. TE-PAI's error bars come from the quasiprobability **overhead** $\gamma$ (the price of shallower circuits), not from shot noise.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from pai_shadow.hamil import Heisenberg_Hamil
from pai_shadow.backend import get_backend
from pai_shadow.trotter import trotter_circuit
from pai_shadow.te_pai import TEPAI

np.random.seed(0)

In [ ]:
# 7-qubit Heisenberg spin chain
n_qubits = 7
H = Heisenberg_Hamil(n_qubits, 1.0, 1.0, 1.0)
backend = get_backend("qulacs")  # or "qiskit"

# Neel initial state |0101010>
neel = [q % 2 for q in range(n_qubits)]
idx = sum(b << q for q, b in enumerate(neel))
psi0 = np.zeros(1 << n_qubits, dtype=complex)
psi0[idx] = 1.0

observable = "Z" + "I" * (n_qubits - 1)   # <Z_0>

# parameters (n_steps large enough that 2|coef|*dt <= delta for TE-PAI)
n_steps = 40
delta   = np.pi / 32
M       = 8000   # TE-PAI circuits (evaluated in parallel)
Ns      = 3000   # Trotter measurement shots
times   = np.linspace(0.0, 1.5, 11)

def z0(bitstring):
    return 1 - 2 * int(bitstring[-1])  # Z eigenvalue on qubit 0 (right-most bit)

In [ ]:
exact, trot_mean, trot_err, tepai_mean, tepai_err = [], [], [], [], []
for t in times:
    circ = trotter_circuit(H, t, n_steps, init_state=psi0)
    exact.append(backend.expectation(circ, observable))
    # Trotter with finite measurement shots
    v = np.array([z0(b) for b in backend.sample(circ, Ns)])
    trot_mean.append(v.mean()); trot_err.append(v.std() / np.sqrt(Ns))
    # TE-PAI: M random circuits, evaluated in parallel across all cores
    tp = TEPAI(H, delta, t, n_steps, init_state=psi0)
    w = tp.estimate(observable, M, backend="qulacs", n_jobs=os.cpu_count())
    tepai_mean.append(w.mean()); tepai_err.append(w.std() / np.sqrt(M))
    print(f"t={t:.2f}  exact={exact[-1]:+.3f}  overhead={tp.overhead:.2f}")

In [ ]:
plt.figure(figsize=(9, 5))
k = 2  # plot 95% confidence (+/- 2 sigma)
plt.plot(times, exact, "k-", lw=2, label="exact (statevector)")
plt.errorbar(times, trot_mean, yerr=k*np.array(trot_err), fmt="o", capsize=3,
             color="tab:blue", label=f"Trotter, {Ns} shots")
plt.errorbar(times, tepai_mean, yerr=k*np.array(tepai_err), fmt="s", capsize=3,
             color="tab:red", label=f"TE-PAI, M={M}")
plt.axhline(0, color="gray", lw=0.5)
plt.xlabel("time $t$"); plt.ylabel(r"$\langle Z_0(t)\rangle$")
plt.title(r"7-qubit Heisenberg: Trotter vs TE-PAI ($\pm2\sigma$ error bars)")
plt.legend(); plt.tight_layout(); plt.show()

## Notes

- `TEPAI.estimate(...)` fuses circuit **generation + evaluation inside worker processes** and splits the work over `n_jobs` cores (a persistent pool is reused across the time points, so process startup is paid only once).
- It returns the per-circuit weighted values; the **mean** is the unbiased estimate and **std/√M** the error bar.
- Pass `shots=<int>` to `estimate` for literal single-shot measurement snapshots (Z/I observables); the default `shots=None` uses each circuit's exact expectation (lower variance, faster convergence).
- Lower `delta` or raise `n_steps` to see how `overhead` and the TE-PAI error bars change.
- Error bars show **95% confidence ($\pm2\sigma$)**. With $\pm1\sigma$ bars the exact value lies inside only ~68% of the points *by definition* — that is expected, not a bug.

## Robustness to gate noise

Now add **depolarizing (or other) gate noise** after each rotation. The deep Trotter circuit
accumulates much more error than the shallow TE-PAI circuits, so as the two-qubit error rate
$p_2$ grows the Trotter estimate **collapses toward 0** while TE-PAI **stays close to the exact
value** — the key advantage demonstrated in the paper.

We probe at a small time where $\langle Z_0\rangle$ is large (so the noise-induced decay is
easy to see). Change `noise_kind` to explore different channels.

In [ ]:
from pai_shadow.backend import NoiseSpec

noise_kind   = "depolarizing"   # try: "bitflip", "phaseflip", "amplitude_damping"
t_noise      = 0.2              # small t -> <Z_0> is large -> decay is visible
n_steps_noise = 60              # deep Trotter circuit
M_noise, Ns_noise = 5000, 5000
p2_values = np.array([0.0, 2e-3, 5e-3, 1e-2, 2e-2, 4e-2])

tcirc = trotter_circuit(H, t_noise, n_steps_noise, init_state=psi0)
exact_z = backend.expectation(tcirc, observable)   # noiseless reference
print(f"exact <Z0> = {exact_z:+.3f}   Trotter depth = {tcirc.depth()}")

In [ ]:
tro_m, tro_e, tep_m, tep_e = [], [], [], []
for p2 in p2_values:
    ns = NoiseSpec(p1=p2 / 10, p2=p2, kind=noise_kind)
    noisy = get_backend("qulacs", noise=ns)
    # Trotter: one deep circuit, many noisy shots
    v = np.array([z0(b) for b in noisy.sample(tcirc, Ns_noise)])
    tro_m.append(v.mean()); tro_e.append(v.std() / np.sqrt(Ns_noise))
    # TE-PAI: many shallow circuits, one noisy shot each (weighted)
    tp = TEPAI(H, delta, t_noise, n_steps_noise, init_state=psi0)
    w = tp.estimate(observable, M_noise, shots=1, n_jobs=os.cpu_count(), noise=ns)
    tep_m.append(w.mean()); tep_e.append(w.std() / np.sqrt(M_noise))
    print(f"p2={p2:.0e}  Trotter={tro_m[-1]:+.3f}  TE-PAI={tep_m[-1]:+.3f}")

In [ ]:
plt.figure(figsize=(8, 5))
plt.axhline(exact_z, color="black", ls="--", lw=1.5, label="exact (noiseless)")
plt.errorbar(p2_values, tro_m, yerr=2*np.array(tro_e), fmt="o-", capsize=3,
             color="tab:blue", label="Trotter (deep)")
plt.errorbar(p2_values, tep_m, yerr=2*np.array(tep_e), fmt="s-", capsize=3,
             color="tab:red", label="TE-PAI (shallow)")
plt.xlabel("two-qubit error rate $p_2$"); plt.ylabel(r"$\langle Z_0\rangle$")
plt.title(f"Noise robustness ({noise_kind}): TE-PAI vs Trotter at $t={t_noise}$")
plt.legend(); plt.tight_layout(); plt.show()

### Notes on the noisy comparison

- Noise enters only through **measurement sampling** (`shots`), which is why TE-PAI is run with `shots=1` and a `NoiseSpec` here (exact expectation is always noiseless).
- TE-PAI's larger error bars are the quasiprobability overhead; its **bias** stays small because each sampled circuit is shallow.
- Try `noise_kind = "bitflip" / "phaseflip" / "amplitude_damping"`, or raise `n_steps_noise` to make the Trotter circuit deeper and the contrast sharper.